# 10 - Advanced Feature Engineering

## Objective

Convert the cleaned and transformed dataset into a **model-ready feature matrix**.

This notebook demonstrates feature engineering techniques commonly used in
industry MMM projects before model training.

### Topics
- Interaction features
- Marketing efficiency metrics
- Spend share
- Price elasticity proxies
- Polynomial features
- Variance Inflation Factor (VIF)
- Mutual Information
- Recursive Feature Elimination (RFE)
- Final feature selection


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_regression,RFE
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.stats.outliers_influence import variance_inflation_factor

ROOT=Path.cwd()
DATA=ROOT/"data"/"processed"/"marketing_mix_features.csv"

df=pd.read_csv(DATA,parse_dates=["Week"])

media=[
"Google_Search","Google_Display","Meta","Instagram",
"YouTube","TV","Radio","Influencer","Affiliate","Email"
]


## 1. Total Marketing Spend

In [ ]:

df["Total_Media_Spend"]=df[media].sum(axis=1)

for col in media:
    df[col+"_Share"]=df[col]/df["Total_Media_Spend"]

df.filter(regex="Share").head()


## 2. Marketing Interaction Features

In [ ]:

df["TV_Holiday"]=df["TV"]*df["Holiday"]
df["Meta_Discount"]=df["Meta"]*df["Discount"]
df["Search_Price"]=df["Google_Search"]*df["Price"]
df["Competitor_vs_Search"]=df["Competitor_Spend"]/(df["Google_Search"]+1)

df[[
"TV_Holiday",
"Meta_Discount",
"Search_Price"
]].head()


## 3. Price Elasticity Proxy

In [ ]:

df["Price_Change"]=df["Price"].pct_change().fillna(0)
df["Sales_Change"]=df["Sales"].pct_change().fillna(0)

df["Elasticity_Proxy"]=(
df["Sales_Change"]/
(df["Price_Change"]+1e-6)
)

df[["Price_Change","Sales_Change","Elasticity_Proxy"]].head()


## 4. Polynomial Features

In [ ]:

poly=PolynomialFeatures(
degree=2,
include_bias=False
)

poly_df=pd.DataFrame(
poly.fit_transform(df[["Google_Search","Meta"]]),
columns=poly.get_feature_names_out(["Google_Search","Meta"])
)

display(poly_df.head())


## 5. Variance Inflation Factor

In [ ]:

vif_features=[
"Google_Search_Adstock",
"Meta_Adstock",
"TV_Adstock",
"Trend",
"Sales_MA_12"
]

X=df[vif_features].fillna(0)

vif=pd.DataFrame({
"Feature":X.columns,
"VIF":[variance_inflation_factor(X.values,i)
for i in range(X.shape[1])]
})

display(vif.sort_values("VIF",ascending=False))


## 6. Mutual Information

In [ ]:

candidate=[
"Google_Search_Hill",
"Meta_Hill",
"TV_Hill",
"Trend",
"Sales_MA_12",
"Sales_Lag_4",
"Discount",
"Holiday",
"Competitor_Spend"
]

mi=mutual_info_regression(
df[candidate].fillna(0),
df["Sales"]
)

mi_df=pd.DataFrame({
"Feature":candidate,
"Mutual_Information":mi
}).sort_values("Mutual_Information",ascending=False)

display(mi_df)


## 7. Recursive Feature Elimination

In [ ]:

model=LinearRegression()

selector=RFE(
estimator=model,
n_features_to_select=6
)

selector.fit(
df[candidate].fillna(0),
df["Sales"]
)

rfe=pd.DataFrame({
"Feature":candidate,
"Selected":selector.support_,
"Rank":selector.ranking_
})

display(rfe.sort_values("Rank"))


## 8. Final Model Feature List

In [ ]:

final_features=[
"Google_Search_Hill",
"Meta_Hill",
"TV_Hill",
"Trend",
"Sales_MA_12",
"Sales_Lag_4",
"Discount",
"Holiday",
"Competitor_Spend",
"Price",
"Temperature",
"Quarter"
]

feature_matrix=df[final_features+["Sales"]]

display(feature_matrix.head())

OUT=ROOT/"data"/"processed"/"marketing_mix_model_ready.csv"
feature_matrix.to_csv(OUT,index=False)

print("Saved:",OUT)
print("Shape:",feature_matrix.shape)


# Business Summary

By this stage we have engineered a feature set suitable for traditional MMM.

## Typical Production Pipeline

1. Raw Media
2. Adstock
3. Hill Saturation
4. Calendar Features
5. Trend & Seasonality
6. Interaction Features
7. Feature Selection
8. Train MMM

## Interview Questions

1. Why create interaction features?
2. What is multicollinearity and why does VIF matter?
3. Why use Mutual Information instead of only correlation?
4. Why perform feature selection before regression?
5. Which engineered features would you expect to have the strongest business impact?

## Next Notebook

**11_MMM_Model_Development.ipynb**

We'll build and compare:
- Multiple Linear Regression
- Ridge Regression
- Lasso Regression
- ElasticNet
- Performance comparison
- Coefficient interpretation
- Contribution estimation
